In [2]:
# 🚀 DAY 37: GPU BATCH SCALING → 100K TICKS
print("🎯 RTX 4070 Ti: 1000→100K ticks | Day36 47ms baseline")
print("Batch scaling = HFT production capacity")

import torch
import torch.nn as nn
import time
import numpy as np
import pandas as pd

device = torch.device('cuda')

# Day31 LSTM as in Day36
class RegimeLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(20, 64, 2, batch_first=True)
        self.regime_head = nn.Linear(64, 3)

    def forward(self, x):
        out, _ = self.lstm(x)
        return torch.softmax(self.regime_head(out[:, -1]), dim=-1)

model = RegimeLSTM().to(device).eval()

batch_sizes = [1000, 10000, 50000, 100000]
results = []

for batch_size in batch_sizes:
    # Generate batch: (batch, seq_len=20) → unsqueeze to (batch, 1, 20)
    ticks = np.random.randn(batch_size, 20).astype(np.float32)
    x = torch.from_numpy(ticks).unsqueeze(0).to(device)   # (1, B, 20)

    # Sync GPU, time inference
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        regimes = model(x)
    torch.cuda.synchronize()
    end = time.perf_counter()

    total_ms = (end - start) * 1000
    ms_per_tick = total_ms / batch_size
    gb_usage = torch.cuda.max_memory_allocated(device) / 1e9
    torch.cuda.reset_peak_memory_stats(device)

    results.append((batch_size, total_ms, ms_per_tick, gb_usage))
    print(f"Batch {batch_size:,}: {total_ms:.1f}ms → {ms_per_tick:.3f}ms/tick | {gb_usage:.1f}GB")

df = pd.DataFrame(results, columns=['Batch', 'Total_ms', 'ms_per_tick', 'GB'])
print("\n✅ LINEAR SCALING:")
print(df)

print("\n🎯 DAY 37: Production GPU capacity!")
print("• GitHub Day37 → 37/90")

🎯 RTX 4070 Ti: 1000→100K ticks | Day36 47ms baseline
Batch scaling = HFT production capacity
Batch 1,000: 41.0ms → 0.041ms/tick | 0.0GB
Batch 10,000: 9.4ms → 0.001ms/tick | 0.1GB
Batch 50,000: 40.2ms → 0.001ms/tick | 0.3GB


RuntimeError: cuDNN error: CUDNN_STATUS_NOT_SUPPORTED. This error may appear if you passed in a non-contiguous input.

In [3]:
# 🚀 DAY 37: GPU BATCH SCALING → 100K TICKS
print("🎯 RTX 4070 Ti: 1000→100K ticks | Day36 47ms baseline")
print("Batch scaling = HFT production capacity")

import torch
import torch.nn as nn
import time
import numpy as np
import pandas as pd

device = torch.device('cuda')

# Day31 LSTM as in Day36
class RegimeLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(20, 64, 2, batch_first=True)
        self.regime_head = nn.Linear(64, 3)

    def forward(self, x):
        out, _ = self.lstm(x)
        return torch.softmax(self.regime_head(out[:, -1]), dim=-1)

model = RegimeLSTM().to(device).eval()

batch_sizes = [1000, 10000, 50000, 100000]
results = []

for batch_size in batch_sizes:
    # Generate batch: (B, 20) → (B, 1, 20) with contiguous layout
    ticks = np.random.randn(batch_size, 20).astype(np.float32)
    x = torch.from_numpy(ticks).unsqueeze(1).to(device).contiguous()   # (B, 1, 20)

    # Sync GPU, time inference
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        regimes = model(x)   # now safe for cuDNN
    torch.cuda.synchronize()
    end = time.perf_counter()

    total_ms = (end - start) * 1000
    ms_per_tick = total_ms / batch_size
    gb_usage = torch.cuda.max_memory_allocated(device) / 1e9
    torch.cuda.reset_peak_memory_stats(device)

    results.append((batch_size, total_ms, ms_per_tick, gb_usage))
    print(f"Batch {batch_size:,}: {total_ms:.1f}ms → {ms_per_tick:.3f}ms/tick | {gb_usage:.1f}GB")

df = pd.DataFrame(results, columns=['Batch', 'Total_ms', 'ms_per_tick', 'GB'])
print("\n✅ LINEAR SCALING:")
print(df)

print("\n🎯 DAY 37: Production GPU capacity!")
print("• GitHub Day37 → 37/90")

🎯 RTX 4070 Ti: 1000→100K ticks | Day36 47ms baseline
Batch scaling = HFT production capacity
Batch 1,000: 5.3ms → 0.005ms/tick | 0.0GB
Batch 10,000: 28.7ms → 0.003ms/tick | 0.1GB
Batch 50,000: 168.9ms → 0.003ms/tick | 0.6GB
Batch 100,000: 134.3ms → 0.001ms/tick | 1.2GB

✅ LINEAR SCALING:
    Batch  Total_ms  ms_per_tick        GB
0    1000    5.2581     0.005258  0.042797
1   10000   28.7041     0.002870  0.137128
2   50000  168.9139     0.003378  0.615156
3  100000  134.3013     0.001343  1.213036

🎯 DAY 37: Production GPU capacity!
• GitHub Day37 → 37/90
